# 発展：フーリエ変換とFFT

第14回では、22年周期を一つ選び、

$$
A\sin\left(\frac{2\pi x}{T}\right)
+
B\cos\left(\frac{2\pi x}{T}\right)
$$

をデータに当てはめて、その周期の変動を取り出しました。

しかし実際の時系列には、さまざまな周期の変動が同時に含まれています。

ここでは、多数の周期をまとめて調べる**フーリエ変換**と、それを高速に計算する**FFT（Fast Fourier Transform; 高速フーリエ変換）**を扱います。

このNotebookは**発展内容**です。Codeセルは基本的に上から順番に実行してください。

## 1. 複数の周期を含むデータを作る

まず、周期8と周期20の2つの波を重ねたデータを作ります。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 128
t = np.arange(N)

y1 = 2.0*np.sin(2*np.pi*t/8)
y2 = 1.0*np.sin(2*np.pi*t/20)
y = y1 + y2

plt.plot(t, y, label="y")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.show()

図だけを見ても周期的な変動があることは分かりますが、周期8と周期20が重なっていることを正確に読み取るのは簡単ではありません。

そこで、「どの周期の波がどのくらい含まれているか」を調べます。

## 2. 第14回の調和解析をいろいろな周期に繰り返す

第14回では、ある周期 $T$ に対して

$$
A(T)=
\frac{\sum_i y_i\sin(2\pi t_i/T)}
{\sum_i\sin^2(2\pi t_i/T)}
$$

$$
B(T)=
\frac{\sum_i y_i\cos(2\pi t_i/T)}
{\sum_i\cos^2(2\pi t_i/T)}
$$

を計算しました。

そして、その周期の振幅は

$$
\sqrt{A(T)^2+B(T)^2}
$$

でした。

同じ計算を、いろいろな周期について行ってみます。

In [ ]:
periods = np.linspace(2.1, 40, 500)
amps = []

for T in periods:
    s = np.sin(2*np.pi*t/T)
    c = np.cos(2*np.pi*t/T)

    A = np.sum(y*s) / np.sum(s**2)
    B = np.sum(y*c) / np.sum(c**2)

    amps.append(np.sqrt(A**2 + B**2))

plt.plot(periods, amps)
plt.xlabel("Period")
plt.ylabel("Amplitude")
plt.show()

周期8付近と周期20付近で振幅が大きくなります。

つまり、時系列をさまざまな周期のsin・cos関数と比較することで、データに含まれる周期を調べることができます。

これがフーリエ解析の基本的な考え方です。

## 3. さまざまな周期の波に分解する

第14回PDFで扱った考え方を一般化すると、データ $f_i$ はさまざまな周期のsin・cos成分の和として

$$
f_i
=
\sum_n
\left[
A_n\sin\left(\frac{n\pi x_i'}{N}\right)
+
B_n\cos\left(\frac{n\pi x_i'}{N}\right)
\right]
$$

のように表すことができます。

それぞれの $n$ について $A_n$, $B_n$ を求めれば、各周期の成分を取り出せます。

異なる周期のsin・cos関数どうしは、データ全体で足し合わせると互いに打ち消し合う性質を持っています。この性質を利用すると、各周期の係数を独立に求めることができます。

第14回PDFでは、係数を

$$
A_n=
\frac{2}{N}
\sum_i
f_i\sin\left(\frac{n\pi x_i'}{N}\right)
$$

$$
B_n=
\frac{2}{N}
\sum_i
f_i\cos\left(\frac{n\pi x_i'}{N}\right)
$$

の形で求めました。

これは、第14回で一つの周期について行った最小二乗法を、多数の周期へ広げたものと考えることができます。

## 4. NumPyでフーリエ変換を計算する

実際には、すべての周期についてsin・cosを一つずつ計算する代わりに、FFTを使うとまとめて高速に計算できます。

NumPyでは `np.fft.rfft()` を使います。

In [ ]:
F = np.fft.rfft(y)
freq = np.fft.rfftfreq(N, d=1.0)

amplitude = 2*np.abs(F)/N

plt.plot(freq[1:], amplitude[1:])
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.show()

横軸は**周波数**です。

周波数 $f$ と周期 $T$ には

$$
T=\frac{1}{f}
$$

という関係があります。

例えば、

- 周期8 → 周波数 $1/8$
- 周期20 → 周波数 $1/20$

です。

In [ ]:
period = 1 / freq[1:]

plt.plot(period, amplitude[1:])
plt.xlim(40, 2)
plt.xlabel("Period")
plt.ylabel("Amplitude")
plt.show()

周期を横軸にすると、もとのデータに入れた周期8と周期20付近に大きな成分があることが分かります。

## 5. FFTは何を返しているのか

`np.fft.rfft()` が返す値は複素数です。

In [ ]:
print(F[:10])

複素数を

$$
F_k=a_k+i b_k
$$

とすると、その大きさは

$$
|F_k|=\sqrt{a_k^2+b_k^2}
$$

です。

これは、第14回で求めた

$$
\sqrt{A^2+B^2}
$$

と対応する考え方です。

実部と虚部には、cos成分とsin成分に対応する情報がまとめて入っています。したがって、FFTの結果から**振幅だけでなく位相**も求めることができます。

## 6. 周期ごとの変動の大きさ

第14回PDFでは、周期成分

\[
A_n\sin(\cdots)+B_n\cos(\cdots)
\]

の平均二乗が

\[
\frac{A_n^2+B_n^2}{2}
\]

になることを扱いました。

したがって、`A_n^2 + B_n^2`、あるいはFFT係数の大きさの2乗を見ると、その周期がデータの変動にどれくらい寄与しているかを調べることができます。

In [ ]:
power = np.abs(F)**2

plt.plot(freq[1:], power[1:])
plt.xlabel("Frequency")
plt.ylabel("Power")
plt.show()

このように、周波数ごとの変動の大きさを表したものを**スペクトル**と呼びます。

時系列そのものを見るのが「時間領域」での見方だとすれば、FFTは同じデータを「周波数領域」から見る方法です。

## 7. 実際のデータで使うときの注意

FFTは便利ですが、どのようなデータにもそのまま適用すればよいわけではありません。

特に、

- データが等間隔に並んでいるか
- 長期トレンドが残っていないか
- 解析期間の長さが十分か
- 解析区間の端で値が不連続になっていないか

などによって、得られるスペクトルは変わります。

実際の観測データを解析するときには、これらを確認したうえで結果を解釈する必要があります。

## 8. まとめ

第14回では、一つの周期を決めてsin・cos関数を当てはめました。

フーリエ変換では、その考え方を多数の周期へ広げ、

**「この時系列には、どの周期の変動がどのくらい含まれているか」**

を調べます。

FFTは、そのフーリエ変換を効率よく計算するアルゴリズムです。

したがって、

**第14回の調和解析 → 多数の周期の調和解析 → フーリエ変換 → FFT**

というつながりで考えると理解しやすくなります。